# Physical-space wavefunction
Plot $|\psi|^2$, $\mathrm{Re}(\psi)$, $\mathrm{Im}(\psi)$, and $\arg(\psi)$ for one frame, several frames, or a frame average. In average mode the intensity is $\langle|\psi|^2\rangle$ while the other panels use $\langle\psi\rangle$. The output is a PDF.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'gp2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from gp2d_plotting import (discover_wavefunctions, frame_time, parameter_float,
    read_csv, read_parameters, read_wavefunction, repository_root,
    save_figure, select_frames, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()  # Repository root; normally no change is needed.
DATA_DIRECTORY = ROOT / 'data'  # Directory containing wavefunction_XXXXXXXX.dat files.
PARAMETER_FILE = ROOT / 'output/resolved_parameters.txt'  # Supplies the physical domain size.
DIAGNOSTICS_FILE = ROOT / 'output/diagnostics.csv'  # Supplies time labels when available.
INTENSITY_FIGURE = ROOT / 'figures/wavefunction_intensity.pdf'  # |psi|^2 output.
REAL_FIGURE = ROOT / 'figures/wavefunction_real.pdf'  # Re(psi) output.
IMAGINARY_FIGURE = ROOT / 'figures/wavefunction_imaginary.pdf'  # Im(psi) output.
PHASE_FIGURE = ROOT / 'figures/wavefunction_phase.pdf'  # arg(psi) output.

# 'single': one panel and exactly one selected frame.
# 'multiple': one panel for every selected frame.
# 'average': one panel averaged over all selected frames.
MODE = 'single'

# Explicit frame numbers to plot. Negative indices count from the end, so [-1]
# means the most recent frame. Set this to None to use START/STOP/STRIDE below.
FRAMES = [-1]
FRAME_START = None  # First frame when FRAMES=None; None means the first available.
FRAME_STOP = None   # Last frame, inclusive; None means the last available.
FRAME_STRIDE = 1    # Keep every nth available frame in the selected range.

PANEL_COLUMNS = 3   # Maximum number of columns in a multiple-frame figure.
INTERPOLATION = 'bilinear'  # imshow interpolation; use 'nearest' for raw grid cells.
USE_TEX = True      # True uses an external LaTeX installation for all figure text.
FONT_SIZE = 15      # Base font size in points.

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
files = discover_wavefunctions(DATA_DIRECTORY)
frames = select_frames(files, FRAMES, start=FRAME_START, stop=FRAME_STOP, stride=FRAME_STRIDE)
if MODE == 'single' and len(frames) != 1:
    raise ValueError('single mode requires exactly one selected frame')

parameters = read_parameters(PARAMETER_FILE) if PARAMETER_FILE.exists() else {}
aspect_ratio = parameter_float(parameters, 'aspectRatio', 1.0)
lx = parameter_float(parameters, 'domainLengthX', 2.0 * np.pi * aspect_ratio)
ly = parameter_float(parameters, 'domainLengthY', 2.0 * np.pi)
extent = (0.0, lx, 0.0, ly)
diagnostics = read_csv(DIAGNOSTICS_FILE) if DIAGNOSTICS_FILE.exists() else None

if MODE == 'average':
    mean_psi = None
    mean_intensity = None
    for frame in frames:
        psi = read_wavefunction(files[frame])
        mean_psi = psi.copy() if mean_psi is None else mean_psi + psi
        intensity = np.abs(psi) ** 2
        mean_intensity = intensity if mean_intensity is None else mean_intensity + intensity
    mean_psi /= len(frames)
    mean_intensity /= len(frames)
    records = [(f'average of {len(frames)} frames', mean_psi, mean_intensity)]
else:
    records = []
    for frame in frames:
        psi = read_wavefunction(files[frame])
        label = f'frame {frame}'
        if diagnostics is not None and frame in diagnostics['frame']:
            label += rf', $t={frame_time(diagnostics, frame):.4g}$'
        records.append((label, psi, np.abs(psi) ** 2))

print(f'Selected frames: {frames}')

In [ ]:
real_limit = max(np.max(np.abs(psi.real)) for _, psi, _ in records)
imag_limit = max(np.max(np.abs(psi.imag)) for _, psi, _ in records)
component_limit = max(real_limit, imag_limit, np.finfo(float).eps)
intensity_limit = max(max(np.max(intensity) for _, _, intensity in records),
                      np.finfo(float).eps)

def plot_quantity(labelled_fields, title, cmap, vmin, vmax, destination):
    count = len(labelled_fields)
    columns = min(PANEL_COLUMNS, count)
    rows = int(np.ceil(count / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(5.0 * columns, 4.3 * rows),
                             squeeze=False)
    flat_axes = list(axes.flat)
    for axis, (frame_label, values) in zip(flat_axes, labelled_fields):
        image = axis.imshow(values, origin='lower', extent=extent,
                            interpolation=INTERPOLATION, cmap=cmap, vmin=vmin, vmax=vmax)
        axis.set_title(title + '\n' + frame_label)
        axis.set_xlabel(r'$x$')
        axis.set_ylabel(r'$y$')
        axis.set_aspect('equal')
    for axis in flat_axes[count:]:
        axis.set_visible(False)
    fig.colorbar(image, ax=flat_axes[:count], shrink=0.85, pad=0.03)
    saved = save_figure(fig, destination)
    print(f'Wrote {saved}')
    return fig

## Wavefunction intensity

In [ ]:
fig = plot_quantity([(label, intensity) for label, _, intensity in records],
                    r'$|\psi|^2$', 'magma', 0.0, intensity_limit, INTENSITY_FIGURE)
plt.show()

## Real part

In [ ]:
fig = plot_quantity([(label, psi.real) for label, psi, _ in records],
                    r'$\mathrm{Re}(\psi)$', 'RdBu_r', -component_limit, component_limit,
                    REAL_FIGURE)
plt.show()

## Imaginary part

In [ ]:
fig = plot_quantity([(label, psi.imag) for label, psi, _ in records],
                    r'$\mathrm{Im}(\psi)$', 'RdBu_r', -component_limit, component_limit,
                    IMAGINARY_FIGURE)
plt.show()

## Phase

In [ ]:
fig = plot_quantity([(label, np.angle(psi)) for label, psi, _ in records],
                    r'$\arg(\psi)$', 'twilight', -np.pi, np.pi, PHASE_FIGURE)
plt.show()